# 02 - Análise Exploratória de Dados (EDA)

Este notebook realiza a Análise Exploratória de Dados (EDA) da base de notas fiscais do projeto `model_supermarket`, utilizando os módulos analíticos desenvolvidos na arquitetura do framework.

O objetivo é explorar padrões de consumo, comportamento temporal, distribuição de gastos, frequência de produtos e indicadores gerais das compras realizadas em supermercados.

As análises incluem:

- validação estrutural da base
- resumos estatísticos
- métricas gerais de consumo
- análises temporais
- agregações por supermercado
- análises por produto
- histórico de preços
- identificação de padrões de compra
- preparação para visualização em dashboard e modelos estatísticos

In [15]:
# !pip install -r requirements.txt

In [16]:
# IMPORTS
from pathlib import Path
import time
import openpyxl
import pandas as pd
from pandas import read_excel
from src.core.config_loader import ConfigLoader
from src.core.tictoc import tictoc
from src.analysis.eda import *

# CONFIGURAÇÃO
tic_geral = time.time()
config = ConfigLoader("config/config.yaml")
raw_path = Path(config.get('data.raw_invoice_path'))

## 1. Carregamento dos Dados

| Variável               | Tipo da Variável           | Descrição                                                                   |
| ---------------------- | -------------------------- | --------------------------------------------------------------------------- |
| `CHAVE_ANONIMIZADA`    | Chave                      | Identificador único anonimizado da NFC-e utilizando hash SHA256.            |
| `DATA`                 | Data                       | Data e horário de emissão da nota fiscal no formato `DD/MM/AAAA HH:MM:SS`.  |
| `MES_ANO`              | Categórico                 | Representação da competência da compra no formato `AAAAMM`.                 |
| `PERIODO`              | Categórico                 | Período do dia em que a compra foi realizada (`MANHA`, `TARDE` ou `NOITE`). |
| `CNPJ`                 | Chave                      | CNPJ do estabelecimento emissor da nota fiscal.                             |
| `SUPERMERCADO`         | Categórico                 | Nome do supermercado onde a compra foi realizada.                           |
| `QTDE_TOTAL_NOTA`      | Numérico Inteiro           | Quantidade total de itens presentes na nota fiscal.                         |
| `VALOR_TOTAL_NOTA`     | Numérico Contínuo          | Valor total da nota fiscal.                                                 |
| `VALOR_TOTAL_TRIBUTOS` | Numérico Contínuo          | Valor total estimado de tributos incidentes sobre a compra.                 |
| `COD_PRODUTO`          | Chave                      | Código identificador único do produto.                                      |
| `CAT_PRODUTO`          | Categórico                 | Categoria associada ao produto.                                             |
| `PRODUTO`              | Categórico                 | Nome/descrição padronizada do produto.                                      |
| `UNIDADE`              | Categórico                 | Unidade de medida ou comercialização do produto.                            |
| `QTDE`                 | Numérico Contínuo          | Quantidade adquirida do produto na nota fiscal.                             |
| `VALOR_PRODUTO`        | Numérico Contínuo          | Valor unitário do produto.                                                  |
| `VALOR_TOTAL_PRODUTO`  | Numérico Contínuo          | Valor total pago pelo produto (`QTDE × VALOR_PRODUTO`).                     |


In [17]:
tic = time.time()

In [18]:
df = read_excel('data/notas_fiscais_supermercado.xlsx')
df.head()

,CHAVE_ANONIMIZADA,DATA,MES_ANO,PERIODO,CNPJ,SUPERMERCADO,QTDE_TOTAL_NOTA,VALOR_TOTAL_NOTA,VALOR_TOTAL_TRIBUTOS,COD_PRODUTO,CAT_PRODUTO,PRODUTO,UNIDADE,QTDE,VALOR_PRODUTO,VALOR_TOTAL_PRODUTO
0,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,940360001,CONGELADO,TEKITOS 300G TRAD,UNIDADE,1.0,8.99,8.99
1,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,11112050001,BEBIDA NAO ALCOOLICA,SUCO SUPER 3LT UVA,UNIDADE,1.0,17.90,17.90
2,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,184710001,BEBIDA NAO ALCOOLICA,GATORADE 500ML UVA,GARRAFA,1.0,3.99,3.99
3,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,11516920001,CEREAL,NESCAU ACTIV-GO 370G,UNIDADE,1.0,6.29,6.29
4,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,8290001,BEBIDA NAO ALCOOLICA,CHA LEAO NAT 40G,UNIDADE,1.0,3.49,3.49


In [19]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:00:01) [15/05/2026 04:09:34 - [15/05/2026 04:09:35]


## 2. Análise

**🔹 Resumos Gerais**

* Quantidade total de produtos distintos
* Quantidade total de códigos distintos
* Verificação de inconsistências entre códigos e nomes
* Total gasto em compras
* Quantidade de notas fiscais
* Média de gasto por compra

**🔹 Análises por Categoria Temporal**

* Resumo de gastos por:

  * supermercado
  * período do dia
  * ano
  * mês
* Evolução temporal dos gastos
* Frequência de compras ao longo do tempo

**🔹 Análises por Produto**

* Identificação do produto mais comprado
* Consulta detalhada por código do produto
* Consulta textual por nome do produto
* Histórico de preços dos produtos
* Frequência de compra por item

### 2.1 Resumos Gerais

In [20]:
prod_unicos = produtos_unicos(df)

Antes da transformação:
Há 1016 código(s) de produto(s) distinto(s).
Há 1016 nome(s) de produto(s) distinto(s).


Após a transformação:
Há 1016 código(s) de produto(s) distinto(s).
Há 1016 nome(s) de produto(s) distinto(s).
Há 0 código(s) de produto(s) com nomes diferentes


In [21]:
resumo_val_gerais = resumo_valores_gerais(df)

Resumos gerais:
Total de gastos: R$ 26133.15
Quantidade de idas ao mercado: 138
Média de gastos por nota fiscal: 189.37


### 2.2 Análises por Categoria Temporal

In [22]:
resumo_supermercado, resumo_periodo = resumo_valores_lugar_periodo(df)


Resumo por supermercado:
SUPERMERCADO
ASSAI      6523.10
CONDOR      745.19
MAX       18864.86
Name: VALOR_TOTAL_PRODUTO, dtype: float64

Resumo por período:
PERIODO
MANHA      551.02
NOITE    18498.71
TARDE     7083.42
Name: VALOR_TOTAL_PRODUTO, dtype: float64


In [23]:
resumo_total_por_ano, resumo_total_por_mes = resumo_valores_ano_mes(df)


Resumo por ano:
DATA
2022    3228.52
2023    7041.11
2024    6652.56
2025    6387.64
2026    2823.32
Name: VALOR_TOTAL_PRODUTO, dtype: float64

Resumo por mês:
DATA
2022-06     683.99
2022-07     318.72
2022-08      33.55
2022-09     810.00
2022-10     241.37
2022-11     448.76
2022-12     692.13
2023-01     467.71
2023-02    1095.87
2023-03     470.46
2023-04     436.79
2023-05     169.03
2023-06     686.01
2023-07     942.78
2023-08     856.99
2023-09    1063.81
2023-10     454.55
2023-11      77.17
2023-12     319.94
2024-01    1086.94
2024-02     754.84
2024-03     686.28
2024-04     721.42
2024-05     117.84
2024-06     159.38
2024-07     253.54
2024-08    1031.75
2024-10     438.63
2024-11     878.91
2024-12     523.03
2025-01    1526.36
2025-02    1414.98
2025-03      12.78
2025-05     135.45
2025-08     524.56
2025-09     647.12
2025-10     956.44
2025-11     299.03
2025-12     870.92
2026-01     747.64
2026-02     161.36
2026-03     834.28
2026-04     833.84
2026-05     246.2

### 2.2 Análises por Produto

In [24]:
produto_mais_comprado = resumo_produto_mais_comprado(df)


Resumo do produto mais comprado:
O produto que mais comprei foi: AGUA CRIST AZ 510ML


In [25]:
exibe_subtabela_cod_produto(df, 940360001) # IndexError: index 940360001 is out of bounds for axis 0 with size 997

,COD_PRODUTO,PRODUTO,VALOR_PRODUTO,DATA
0,940360001,TEKITOS 300G TRAD,8.99,2022-06-28 20:44:50
301,940360001,TEKITOS 300G TRAD,8.99,2022-11-09 11:36:52
320,940360001,TEKITOS 300G TRAD,8.99,2022-11-18 21:41:27
370,940360001,TEKITOS 300G TRAD,8.99,2022-12-06 21:20:11
427,940360001,TEKITOS 300G TRAD,8.99,2022-12-18 10:19:36
438,940360001,TEKITOS 300G TRAD,8.99,2023-01-15 18:17:47
617,940360001,TEKITOS 300G TRAD,8.99,2023-03-14 20:38:05
1165,940360001,TEKITOS 300G TRAD,10.49,2023-09-14 18:26:01


In [26]:
exibe_subtabela_nome_produto(df, 'whisky')

,PRODUTO,COD_PRODUTO,VALOR_PRODUTO,DATA
591,WHISKY PASSP 670ML,279691,48.99,2023-03-01 17:50:26
655,WHISKY PASSP 670ML,279691,47.99,2023-04-08 16:59:59
677,WHISKY PASSP 670ML,279691,47.99,2023-04-20 18:34:11
1625,WHISKY PASSP 670ML,279691,61.99,2024-03-06 20:31:25
1782,WHISKY PASSP 670ML,279691,58.99,2024-04-29 16:44:08
1984,WHISKY PASSP 670ML,279691,49.99,2024-08-25 14:20:05
2009,WHISKY PASSP 670ML,279691,57.99,2024-10-18 21:33:44
2090,WHISKY PASSP 670ML,279691,57.99,2024-11-28 17:17:03
2147,WHISKY PASSP 670ML,279691,49.99,2024-12-24 15:25:31
2184,WHISKY PASSP 670ML,279691,57.99,2025-01-06 21:42:46


In [27]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:02) [15/05/2026 04:09:34 - [15/05/2026 04:09:36]


## 3. Tempo Decorrido

In [28]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")


Tempo decorrido no notebook: (00:00:02) [15/05/2026 04:09:34 - [15/05/2026 04:09:36]
